# Concept bottleneck models on the clean SkinCon dataset

This notebook runs on Google Colab with a GPU. It builds the clean SkinCon dataset, tests whether image embeddings
separate the classes inside the rough-set boundary, then trains hard, soft and hybrid concept bottleneck models on an
EfficientNet-B3 concept extractor with the seeds 42, 64 and 128.

Every result is written as a CSV file to `RESULTS_DIR`. Copy that folder to `data/colab` of the repository before
running `inconsistency_analysis.ipynb`. Checkpoints are kept on Drive, so a disconnected session resumes where it
stopped.

**Inputs on Google Drive** (see the README)
- `skincon/annotations_fitzpatrick17k.csv` and `skincon/annotations_ddi.csv`, downloaded from the SkinCon website
  when missing
- `fitzpatrick17k/fitzpatrick17k.csv` and the Fitzpatrick17k images in `fitzpatrick17k/images`
- the DDI images with `ddi_metadata.csv` in `ddi`
- optionally `clean_dataset.csv` from `data/analysis`, which fixes the clean images and splits of the article

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
SEEDS = [42, 64, 128]
BACKBONE_NAME = "efficientnet_b3"
EMBEDDER = "dinov2_vits14"
USE_DDI = True
MAX_VISUAL_SHARE = 0.30          # largest share of the hybrid logit that the visual units may carry on validation
DOWNLOAD_FITZ_IMAGES = False     # download missing Fitzpatrick17k images from the URLs of the metadata file

PROJECT_DIR = "/content/drive/MyDrive/skincon"
SKINCON_DIR = f"{PROJECT_DIR}/skincon"
FITZ_META = f"{PROJECT_DIR}/fitzpatrick17k/fitzpatrick17k.csv"
FITZ_IMAGES = f"{PROJECT_DIR}/fitzpatrick17k/images"
DDI_DIR = f"{PROJECT_DIR}/ddi"
RELEASED_SPLIT = f"{PROJECT_DIR}/clean_dataset.csv"
WORK_DIR = f"{PROJECT_DIR}/work"         # embeddings, checkpoints and trained models
RESULTS_DIR = f"{PROJECT_DIR}/results"   # CSV files, copied to data/colab

## Clean dataset

The cleaning rules are fixed before any model is trained and are applied to all images before the split.

1. Expert exclusions: the SkinCon flag "Do not consider this image" and the Fitzpatrick17k quality control label
   "wrongly labelled".
2. Confirmed duplicates collapse to the copy with the largest pixel count. A pair is a confirmed duplicate when its
   DINOv2 cosine similarity reaches 0.95 and its perceptual hashes differ by at most 10 bits, or when one detector
   alone reaches a strict threshold (cosine 0.99, or 6 bits).
3. Duplicate clusters whose malignancy labels disagree carry a certified label error and are quarantined.
4. Images leave after visual review only, through `manual_exclusions.csv`.
5. Candidate near copies (cosine 0.90, or 10 bits) form groups that stay inside one split.

No rule reads the concepts.

In [ ]:
import os
import io
import sys
import json
import time
import math
import random
import shutil
import hashlib
import urllib.request
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from scipy.stats import chi2_contingency, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

try:
    import imagehash
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)
    import imagehash

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

SPLIT_SEED = 2024
TAU_GROUP = 0.90
TAU_DUPLICATE = 0.95
TAU_NEAR_EXACT = 0.99
PHASH_CONFIRM_BITS = 10
PHASH_EXACT_BITS = 6
OUTLIER_NEIGHBORS = 5
OUTLIER_REVIEW = 100
MIN_TRAIN_POSITIVES = 30         # a concept enters the vocabulary with at least 30 positive training images
LOCAL_IMAGE_DIR = "/content/skincon_images"
IO_THREADS = 32
EXCLUDE_COLUMN = "Do not consider this image"

SPLIT_DIR = os.path.join(WORK_DIR, "splits")
CLEAN_FILE = os.path.join(SPLIT_DIR, "skincon_clean.csv")
EMBED_FILE = os.path.join(SPLIT_DIR, f"embeddings_{EMBEDDER}.npz")
MANUAL_EXCLUSIONS = os.path.join(SPLIT_DIR, "manual_exclusions.csv")      # image_id
DUPLICATE_DECISIONS = os.path.join(SPLIT_DIR, "duplicate_decisions.csv")  # image_id_a, image_id_b, decision
F17KC_FILE = os.path.join(PROJECT_DIR, "fitzpatrick17k", "Fitzpatrick17k-C.csv")   # optional, comparison only
EARLIER_SPLIT = os.path.join(SPLIT_DIR, "earlier_split.csv")                       # optional, comparison only
for d in (SPLIT_DIR, RESULTS_DIR, SKINCON_DIR):
    os.makedirs(d, exist_ok=True)
for name in ("annotations_fitzpatrick17k.csv", "annotations_ddi.csv"):
    if not os.path.exists(os.path.join(SKINCON_DIR, name)):
        urllib.request.urlretrieve(f"https://skincon-dataset.github.io/files/{name}", os.path.join(SKINCON_DIR, name))


def result(name):
    return os.path.join(RESULTS_DIR, name)


def concept_key(name):
    key = "".join(ch.lower() if ch.isalnum() else "_" for ch in name)
    while "__" in key:
        key = key.replace("__", "_")
    return key.strip("_")


def download_fitz_images(df, target_dir, timeout=20):
    import requests
    os.makedirs(target_dir, exist_ok=True)
    ok = failed = 0
    for _, r in df.iterrows():
        dst = os.path.join(target_dir, r["ImageID"])
        if os.path.exists(dst):
            continue
        try:
            resp = requests.get(r["url"], timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            Image.open(io.BytesIO(resp.content)).convert("RGB").save(dst, quality=95)
            ok += 1
        except Exception:
            failed += 1
    print(f"Fitzpatrick17k download: {ok} new images, {failed} failures")


def stage_images(files_by_source):
    """Copy the images from Drive to the local disk once. A zip archive next to an image folder is used if present."""
    import zipfile
    sources = {"fitzpatrick17k": FITZ_IMAGES, "ddi": DDI_DIR}
    found = {}
    for src, files in files_by_source.items():
        dst_dir = os.path.join(LOCAL_IMAGE_DIR, src)
        os.makedirs(dst_dir, exist_ok=True)
        archive = sources[src].rstrip("/") + ".zip"
        todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo and os.path.exists(archive):
            with zipfile.ZipFile(archive) as zf:
                names = {os.path.basename(n): n for n in zf.namelist() if not n.endswith("/")}
                for f in todo:
                    if f in names:
                        with zf.open(names[f]) as a, open(os.path.join(dst_dir, f), "wb") as b:
                            shutil.copyfileobj(a, b)
            todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo:
            on_drive = set(os.listdir(sources[src])) if os.path.isdir(sources[src]) else set()
            copy = [f for f in todo if f in on_drive]
            with ThreadPoolExecutor(IO_THREADS) as ex:
                list(tqdm(ex.map(lambda f: shutil.copyfile(os.path.join(sources[src], f), os.path.join(dst_dir, f)), copy),
                          total=len(copy), desc=f"copying {src} images"))
        local = set(os.listdir(dst_dir))
        found[src] = {f for f in files if f in local}
        print(f"{src}: {len(found[src])} of {len(files)} images available")
    return found


def load_skincon():
    """One row per usable SkinCon image with the label, the 48 concepts, the source and the image path."""
    fa = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_fitzpatrick17k.csv"))
    concepts = [c for c in fa.columns if c not in {"Unnamed: 0", "ImageID", EXCLUDE_COLUMN}]
    meta = pd.read_csv(FITZ_META)
    fa["md5hash"] = fa["ImageID"].astype(str).str.replace(".jpg", "", regex=False)
    f = fa.merge(meta[["md5hash", "three_partition_label", "label", "fitzpatrick_scale", "qc", "url"]],
                 on="md5hash", how="left", validate="one_to_one")
    counts = {"fitzpatrick17k_annotated": len(f)}
    f = f[f[EXCLUDE_COLUMN].fillna(0) == 0]
    counts["fitzpatrick17k_after_exclusion_flag"] = len(f)
    f = f[~f["qc"].astype(str).str.startswith("3")]
    counts["fitzpatrick17k_after_quality_control"] = len(f)
    f = f[f["three_partition_label"].notna()]
    if DOWNLOAD_FITZ_IMAGES:
        download_fitz_images(f, FITZ_IMAGES)
    f = pd.DataFrame({"image_id": "fitz_" + f["md5hash"], "source": "fitzpatrick17k",
                      "file": f["ImageID"].astype(str).values,
                      "malignant": (f["three_partition_label"] == "malignant").astype(int).values,
                      "diagnosis": f["label"].values, "skin_type": f["fitzpatrick_scale"].astype(str).values,
                      **{concept_key(c): f[c].fillna(0).astype(int).values for c in concepts}})
    frames = [f]
    if USE_DDI:
        da = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_ddi.csv"))
        dm = pd.read_csv(os.path.join(DDI_DIR, "ddi_metadata.csv"))
        d = da.merge(dm, left_on="ImageID", right_on="DDI_file", how="left", validate="one_to_one")
        counts["ddi_annotated"] = len(d)
        d = d[(d[EXCLUDE_COLUMN].fillna(0) == 0) & d["malignant"].notna()]
        counts["ddi_after_exclusion_flag"] = len(d)
        malignant = d["malignant"].astype(str).str.lower().isin(["true", "1", "1.0", "yes"]).astype(int)
        d = pd.DataFrame({"image_id": "ddi_" + d["ImageID"].astype(str), "source": "ddi",
                          "file": d["ImageID"].astype(str).values,
                          "malignant": malignant.values, "diagnosis": d["disease"].values,
                          "skin_type": d["skin_tone"].astype(str).values,
                          **{concept_key(c): d[c].fillna(0).astype(int).values for c in concepts}})
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    found = stage_images({s: df.loc[df["source"] == s, "file"].tolist() for s in df["source"].unique()})
    df["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(df["source"], df["file"])]
    exists = pd.Series([f in found[s] for s, f in zip(df["source"], df["file"])], index=df.index)
    counts["images_found"] = int(exists.sum())
    if (~exists).any():
        print(f"{int((~exists).sum())} image files are missing and are dropped")
    df = df[exists].reset_index(drop=True)
    keys = [concept_key(c) for c in concepts]
    return df, keys, dict(zip(keys, concepts)), counts


def image_hashes(paths):
    """Perceptual hash, MD5 checksum and pixel count of every image."""
    def one(p):
        with open(p, "rb") as fh:
            raw = fh.read()
        with Image.open(io.BytesIO(raw)) as im:
            w, h = im.size
            ph = int(str(imagehash.phash(im.convert("RGB"))), 16)
        return ph, hashlib.md5(raw).hexdigest(), w * h

    with ThreadPoolExecutor(IO_THREADS) as ex:
        out = list(tqdm(ex.map(one, paths), total=len(paths), desc="perceptual hashes"))
    return np.array([o[0] for o in out], dtype=np.uint64), [o[1] for o in out], np.array([o[2] for o in out])


def image_embeddings(paths, batch=64):
    """L2-normalized class-token embeddings of DINOv2 ViT-S/14 at 224 x 224 pixels."""
    import torch
    import torchvision.transforms as T
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = torch.hub.load("facebookresearch/dinov2", EMBEDDER).to(device).eval()
    tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    def load(p):
        with Image.open(p) as im:
            return tf(im.convert("RGB"))

    out = []
    with ThreadPoolExecutor(8) as ex, torch.no_grad():
        for i in tqdm(range(0, len(paths), batch), desc="DINOv2 embeddings"):
            x = torch.stack(list(ex.map(load, paths[i:i + batch]))).to(device)
            out.append(torch.nn.functional.normalize(net(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def phash_pairs(ph, max_bits):
    """Pairs (i < j) whose perceptual hashes differ by at most max_bits bits."""
    n = len(ph)
    b = ph.view(np.uint8).reshape(n, 8)
    I, J, D = [], [], []
    for s in range(0, n, 256):
        dist = np.unpackbits(np.bitwise_xor(b[s:s + 256, None, :], b[None, :, :]), axis=-1).sum(-1)
        a, c = np.where(dist <= max_bits)
        a = a + s
        k = a < c
        I.append(a[k]); J.append(c[k]); D.append(dist[a[k] - s, c[k]])
    return np.concatenate(I), np.concatenate(J), np.concatenate(D)


def components(n, I, J):
    """Connected components of the graph with n nodes and edges (I, J)."""
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for a, b in zip(I, J):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return np.array([find(i) for i in range(n)])


def cohen_kappa(a, b):
    po = float(np.mean(a == b))
    pe = float(np.mean(a) * np.mean(b) + (1 - np.mean(a)) * (1 - np.mean(b)))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


def skin_type_value(df):
    v = pd.to_numeric(df["skin_type"], errors="coerce")
    return np.where((df["source"] == "fitzpatrick17k") & (v >= 1), v, np.nan)


def duplicate_report(df, S, pairs_i, pairs_j, concepts):
    """Pairs at the two cosine thresholds, with their disagreements in diagnosis, skin type, malignancy and concepts."""
    fst = skin_type_value(df)
    dx, y, C = df["diagnosis"].values, df["malignant"].values, df[concepts].values
    rows = []
    for tau in (TAU_GROUP, TAU_DUPLICATE):
        k = S[pairs_i, pairs_j] >= tau
        a, b = pairs_i[k], pairs_j[k]
        D = dx[a] != dx[b]
        fd = np.abs(fst[a] - fst[b])
        F1, F2 = np.nan_to_num(fd) >= 1, np.nan_to_num(fd) > 1
        rows.append({"threshold": tau, "pairs": int(k.sum()), "diagnosis differs (D)": int(D.sum()),
                     "FST differs by >= 1 (F1)": int(F1.sum()), "FST differs by > 1 (F2)": int(F2.sum()),
                     "D or F1": int((D | F1).sum()), "D or F2": int((D | F2).sum()),
                     "D and F1": int((D & F1).sum()), "D and F2": int((D & F2).sum()),
                     "malignancy differs": int((y[a] != y[b]).sum()),
                     "concept vector differs": int((C[a] != C[b]).any(axis=1).sum()),
                     "mean concept Hamming distance": float((C[a] != C[b]).sum(axis=1).mean()) if len(a) else np.nan})
    return pd.DataFrame(rows)


def concept_reliability(df, I, J, concepts):
    """Agreement and Cohen's kappa of every concept across confirmed duplicate pairs."""
    rows = []
    for c in concepts:
        a, b = df[c].values[I], df[c].values[J]
        if a.sum() + b.sum() == 0:
            continue
        rows.append({"concept": c, "pairs": len(I), "positive in either": int(((a + b) > 0).sum()),
                     "agreement": float(np.mean(a == b)), "kappa": cohen_kappa(a, b)})
    return pd.DataFrame(rows).sort_values("positive in either", ascending=False)


def add_partition(df):
    meta = pd.read_csv(FITZ_META, usecols=["md5hash", "three_partition_label"])
    part = dict(zip("fitz_" + meta["md5hash"].astype(str), meta["three_partition_label"]))
    df["partition"] = [part.get(i, "malignant" if m == 1 else "benign") if s == "fitzpatrick17k"
                       else ("malignant" if m == 1 else "benign")
                       for i, s, m in zip(df["image_id"], df["source"], df["malignant"])]
    return df


def save_clean_info(counts, vocabulary, concept_names, training_counts):
    pd.Series(counts, name="value").rename_axis("quantity").to_csv(result("clean_counts.csv"))
    pd.DataFrame({"concept": vocabulary, "name": [concept_names[c] for c in vocabulary],
                  "training_images": [training_counts[c] for c in vocabulary]}).to_csv(result("vocabulary.csv"), index=False)
    settings = {"split_seed": SPLIT_SEED, "embedder": EMBEDDER, "tau_group": TAU_GROUP, "tau_duplicate": TAU_DUPLICATE,
                "tau_near_exact": TAU_NEAR_EXACT, "phash_confirm_bits": PHASH_CONFIRM_BITS,
                "phash_exact_bits": PHASH_EXACT_BITS, "min_train_positives": MIN_TRAIN_POSITIVES, "use_ddi": USE_DDI}
    pd.Series(settings, name="value").rename_axis("setting").to_csv(result("cleaning_settings.csv"))


def build_clean():
    df, all_concepts, concept_names, counts = load_skincon()
    df = add_partition(df)
    n = len(df)
    ph, md5, pixels = image_hashes(df["path"].tolist())
    E = image_embeddings(df["path"].tolist())
    np.savez_compressed(EMBED_FILE, image_id=df["image_id"].values.astype(str), emb=E.astype(np.float16))
    S = E @ E.T
    iu, ju = np.triu_indices(n, 1)
    keep = S[iu, ju] >= TAU_GROUP
    si, sj = iu[keep], ju[keep]
    del iu, ju
    hi, hj, hd = phash_pairs(ph, PHASH_CONFIRM_BITS)
    ham = {(a, b): d for a, b, d in zip(hi, hj, hd)}
    first, exact = {}, []
    for i, h in enumerate(md5):
        if h in first:
            exact.append((first[h], i))
        else:
            first[h] = i
    candidates = set(zip(si, sj)) | set(ham) | set(exact)
    manual = {}
    if os.path.exists(DUPLICATE_DECISIONS):
        idx = {v: k for k, v in enumerate(df["image_id"])}
        for r in pd.read_csv(DUPLICATE_DECISIONS).itertuples():
            if r.image_id_a in idx and r.image_id_b in idx:
                a, b = sorted((idx[r.image_id_a], idx[r.image_id_b]))
                manual[(a, b)] = str(r.decision).strip().lower()
                candidates.add((a, b))
    rows = []
    for a, b in sorted(candidates):
        s, d = float(S[a, b]), ham.get((a, b), 64)
        auto = (s >= TAU_NEAR_EXACT) or (d <= PHASH_EXACT_BITS) or (s >= TAU_DUPLICATE and d <= PHASH_CONFIRM_BITS) \
            or ((a, b) in exact)
        confirmed = {"duplicate": True, "different": False}.get(manual.get((a, b)), auto)
        rows.append((a, b, s, d, confirmed, manual.get((a, b), "")))
    P = pd.DataFrame(rows, columns=["i", "j", "cosine", "phash_bits", "confirmed", "manual"])
    P["image_id_a"], P["image_id_b"] = df["image_id"].values[P.i], df["image_id"].values[P.j]
    P["malignant_a"], P["malignant_b"] = df["malignant"].values[P.i], df["malignant"].values[P.j]
    P["diagnosis_a"], P["diagnosis_b"] = df["diagnosis"].values[P.i], df["diagnosis"].values[P.j]
    P.drop(columns=["i", "j"]).to_csv(result("candidate_pairs.csv"), index=False)
    review = P[~P.confirmed & (P.manual == "") & (P.cosine >= TAU_DUPLICATE)].sort_values("cosine", ascending=False)
    review[["image_id_a", "image_id_b", "cosine", "phash_bits"]].assign(decision="").to_csv(
        result("duplicate_review_sheet.csv"), index=False)
    counts["candidate_pairs"] = len(P)
    counts["confirmed_duplicate_pairs"] = int(P.confirmed.sum())
    counts["unconfirmed_pairs_for_review"] = len(review)

    duplicate_report(df, S, si, sj, all_concepts).to_csv(result("duplicate_pairs_by_threshold.csv"), index=False)
    Pc = P[P.confirmed]
    concept_reliability(df, Pc.i.values, Pc.j.values, all_concepts).to_csv(result("concept_reliability_duplicates.csv"), index=False)

    cluster = components(n, Pc.i.values, Pc.j.values)
    df["dup_cluster"], df["pixels"], df["phash"] = cluster, pixels, ph
    g = df.groupby("dup_cluster")
    size = g["image_id"].transform("size")
    df["mal_conflict"] = g["malignant"].transform("nunique") > 1
    df["diagnosis_conflict"] = g["diagnosis"].transform("nunique") > 1
    df["fst_conflict"] = pd.Series(skin_type_value(df)).groupby(df["dup_cluster"]).transform(
        lambda v: (v.max() - v.min()) >= 1 if v.notna().sum() > 1 else False).values
    df["concept_conflict"] = g[all_concepts].transform("nunique").max(axis=1) > 1
    multi = df[size > 1]
    counts["duplicate_clusters"] = int(multi["dup_cluster"].nunique())
    counts["images_in_duplicate_clusters"] = len(multi)
    counts["clusters_conflicting_malignancy"] = int(multi.loc[multi.mal_conflict, "dup_cluster"].nunique())
    counts["clusters_conflicting_diagnosis"] = int(multi.loc[multi.diagnosis_conflict, "dup_cluster"].nunique())
    counts["clusters_conflicting_skin_type"] = int(multi.loc[multi.fst_conflict, "dup_cluster"].nunique())
    counts["clusters_conflicting_concepts"] = int(multi.loc[multi.concept_conflict, "dup_cluster"].nunique())

    # candidate outliers (lowest similarity to the nearest neighbours), listed for visual review only
    S2 = S.copy()
    np.fill_diagonal(S2, -np.inf)
    df["outlier_score"] = (-np.sort(-S2, axis=1)[:, :OUTLIER_NEIGHBORS]).min(axis=1)
    del S2
    df.sort_values("outlier_score").head(OUTLIER_REVIEW)[["image_id", "source", "diagnosis", "malignant", "outlier_score"]].to_csv(
        result("erroneous_image_candidates.csv"), index=False)

    df["status"] = "kept"
    if os.path.exists(MANUAL_EXCLUSIONS):
        excluded = set(pd.read_csv(MANUAL_EXCLUSIONS)["image_id"].astype(str))
        df.loc[df["image_id"].isin(excluded), "status"] = "manual_excluded"
    df.loc[df["mal_conflict"] & (df["status"] == "kept"), "status"] = "quarantined"
    alive = df["status"] == "kept"
    representatives = (df[alive].sort_values(["dup_cluster", "pixels", "image_id"], ascending=[True, False, True])
                       .drop_duplicates("dup_cluster").index)
    df.loc[alive & ~df.index.isin(representatives), "status"] = "duplicate_removed"
    counts["images_manually_excluded"] = int((df.status == "manual_excluded").sum())
    counts["images_quarantined"] = int((df.status == "quarantined").sum())
    counts["duplicates_removed"] = int((df.status == "duplicate_removed").sum())

    if os.path.exists(F17KC_FILE):
        f17kc = set(pd.read_csv(F17KC_FILE, usecols=["md5hash"])["md5hash"].astype(str))
        fz = df["source"] == "fitzpatrick17k"
        df["in_f17kc"] = np.where(fz, df["image_id"].str.replace("fitz_", "", regex=False).isin(f17kc), np.nan)
        pd.crosstab(df.loc[fz, "status"], df.loc[fz, "in_f17kc"].map({1.0: "in F17k-C", 0.0: "not in F17k-C"})).to_csv(
            result("audit_fitzpatrick17k_c.csv"))
    gi = np.concatenate([si, hi, Pc.i.values])
    gj = np.concatenate([sj, hj, Pc.j.values])
    if os.path.exists(EARLIER_SPLIT):
        earlier = pd.read_csv(EARLIER_SPLIT, usecols=["image_id", "split"]).set_index("image_id")["split"]
        sa, sb = df["image_id"].map(earlier).values[gi], df["image_id"].map(earlier).values[gj]
        counts["near_copy_pairs_across_earlier_split"] = int((pd.notna(sa) & pd.notna(sb) & (sa != sb)).sum())

    clean = df[df["status"] == "kept"].copy()
    pos = {k: i for i, k in enumerate(clean.index)}
    m = np.array([a in pos and b in pos for a, b in zip(df.index[gi], df.index[gj])], dtype=bool)
    clean["near_copy_group"] = components(len(clean), [pos[a] for a in df.index[gi][m]], [pos[b] for b in df.index[gj][m]])
    clean = clean.reset_index(drop=True)
    strata = clean["malignant"].astype(str) + "_" + clean["source"]
    folds = np.zeros(len(clean), dtype=int)
    for k, (_, idx) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
                                 .split(clean, strata, clean["near_copy_group"])):
        folds[idx] = k
    clean["split"] = np.where(folds == 0, "test", np.where(folds == 1, "valid", "train"))
    if RELEASED_SPLIT and os.path.exists(RELEASED_SPLIT):
        # use the released images and splits, so that numerical differences of the embeddings cannot change the data
        released = pd.read_csv(RELEASED_SPLIT).set_index("image_id")
        agree = (clean["split"] == clean["image_id"].map(released["split"])).mean()
        print(f"rebuilt clean set equals the released one: {set(released.index) == set(clean['image_id'])}, "
              f"same split for {agree:.1%} of the images")
        clean = df[df["image_id"].isin(released.index)].copy().reset_index(drop=True)
        clean["near_copy_group"] = clean["image_id"].map(released["near_copy_group"]).values
        clean["split"] = clean["image_id"].map(released["split"]).values
        df["status"] = np.where(df["image_id"].isin(released.index), "kept", "duplicate_removed")
    counts["groups_across_splits"] = int((clean.groupby("near_copy_group")["split"].nunique() > 1).sum())
    counts["clean_images"] = len(clean)

    rows = []
    removed = df["status"] != "kept"
    for col in ("malignant", "partition", "source", "skin_type"):
        ct = pd.crosstab(df[col].astype(str), removed)
        p = chi2_contingency(ct)[1] if ct.shape[0] > 1 and ct.shape[1] > 1 else np.nan
        for level, r in ct.iterrows():
            rows.append({"attribute": col, "level": level, "before": int(r.sum()), "removed": int(r.get(True, 0)),
                         "removal_rate": r.get(True, 0) / r.sum(), "chi2_p_value": p})
    pd.DataFrame(rows).to_csv(result("removal_by_subgroup.csv"), index=False)

    train = clean[clean["split"] == "train"]
    vocabulary = sorted([c for c in all_concepts if train[c].sum() >= MIN_TRAIN_POSITIVES], key=lambda c: (-int(train[c].sum()), c))
    save_clean_info(counts, vocabulary, concept_names, {c: int(train[c].sum()) for c in vocabulary})
    columns = ["image_id", "source", "file", "malignant", "diagnosis", "partition", "skin_type", "near_copy_group", "split",
               "dup_cluster", "diagnosis_conflict", "fst_conflict", "concept_conflict", "outlier_score"] + all_concepts
    clean[columns].to_csv(CLEAN_FILE, index=False)
    df[df.status == "quarantined"][["image_id", "source", "malignant", "diagnosis", "dup_cluster"]].to_csv(
        result("quarantine.csv"), index=False)
    df[["image_id", "source", "malignant", "diagnosis", "status", "dup_cluster", "outlier_score"]].to_csv(
        result("decisions.csv"), index=False)
    return clean[columns + ["path"]]


if os.path.exists(CLEAN_FILE) and os.path.exists(result("vocabulary.csv")):
    DATA = pd.read_csv(CLEAN_FILE)
    found = stage_images({s: DATA.loc[DATA["source"] == s, "file"].tolist() for s in DATA["source"].unique()})
    missing = sum(f not in found[s] for s, f in zip(DATA["source"], DATA["file"]))
    if missing:
        raise FileNotFoundError(f"{missing} images of the clean dataset are missing on Drive")
    DATA["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(DATA["source"], DATA["file"])]
else:
    DATA = build_clean()

VOCABULARY = pd.read_csv(result("vocabulary.csv"))
CONCEPTS = VOCABULARY.concept.tolist()
CONCEPT_NAMES = dict(zip(VOCABULARY.concept, VOCABULARY.name))
_z = np.load(EMBED_FILE, allow_pickle=True)
_emb = dict(zip(_z["image_id"], _z["emb"].astype(np.float32)))
EMB = np.stack([_emb[i] for i in DATA["image_id"]])

print(pd.read_csv(result("clean_counts.csv")).to_string(index=False))
print(DATA.groupby(["split", "source", "malignant"]).size().unstack(fill_value=0))
print(f"vocabulary of {len(CONCEPTS)} concepts with at least {MIN_TRAIN_POSITIVES} positive training images")

## Rough sets of the clean dataset and the image-information test

The rough-set quantities are recomputed on the clean data, overall, per split and per source. The test then asks, on
training images only, whether DINOv2 embeddings separate the classes inside the boundary region where the concepts
cannot.

In [ ]:
def rs_table(X, y):
    _, inv = np.unique(np.asarray(X, dtype=np.int8), axis=0, return_inverse=True)
    inv = inv.reshape(-1)
    k = inv.max() + 1
    n1 = np.bincount(inv, weights=y, minlength=k)
    n0 = np.bincount(inv, minlength=k) - n1
    return inv, n0, n1


def rs_stats(X, y):
    y = np.asarray(y, dtype=int)
    inv, n0, n1 = rs_table(X, y)
    n = n0 + n1
    inc = (n0 > 0) & (n1 > 0)
    bnd = inc[inv]
    P, N = y.sum(), len(y) - y.sum()
    lab = n1 / max(P, 1) > n0 / max(N, 1)
    auc = 0.0
    if P and N:
        r = n1 / n
        below = 0.0
        for v in np.unique(r):
            m = r == v
            auc += n1[m].sum() * below + 0.5 * n1[m].sum() * n0[m].sum()
            below += n0[m].sum()
        auc /= P * N
    ratio = np.minimum(n0, n1)[inc] / n[inc]
    return {"images": len(y), "malignant": int(P), "profiles": len(n), "inconsistent_profiles": int(inc.sum()),
            "boundary_images": int(bnd.sum()), "gamma": 1 - bnd.mean(),
            "acc_ceiling": np.maximum(n0, n1).sum() / len(y),
            "ba_ceiling": 0.5 * (n1[lab].sum() / max(P, 1) + n0[~lab].sum() / max(N, 1)),
            "auroc_ceiling": auc, "malignant_in_boundary": bnd[y == 1].mean() if P else np.nan,
            "non_malignant_in_boundary": bnd[y == 0].mean() if N else np.nan,
            "mean_conflict_ratio": ratio.mean() if len(ratio) else 0.0,
            "singleton_share": (n == 1).sum() / len(y)}


rows = []
for split in ("all", "train", "valid", "test"):
    for src in ("all",) + tuple(sorted(DATA["source"].unique())):
        m = np.ones(len(DATA), bool) if split == "all" else (DATA["split"] == split).to_numpy().copy()
        if src != "all":
            m &= (DATA["source"] == src).values
        if m.sum():
            rows.append({"split": split, "source": src, **rs_stats(DATA.loc[m, CONCEPTS].values, DATA.loc[m, "malignant"].values)})
ROUGH = pd.DataFrame(rows)
ROUGH.to_csv(result("rough_set_clean.csv"), index=False)
print(ROUGH.round(4).to_string(index=False))

y_all = DATA["malignant"].values
inv, n0, n1 = rs_table(DATA[CONCEPTS].values, y_all)
DATA["forced_error"] = (n1 / y_all.sum() > n0 / (len(y_all) - y_all.sum()))[inv].astype(int) != y_all
DATA["in_boundary"] = ((n0 > 0) & (n1 > 0))[inv]
fp = DATA[DATA.malignant == 0].groupby("diagnosis").agg(images=("image_id", "size"), in_boundary=("in_boundary", "mean"),
                                                        forced_false_positive=("forced_error", "mean"))
fp[fp.images >= 15].sort_values("forced_false_positive", ascending=False).to_csv(result("forced_false_positives.csv"))

train = DATA["split"].values == "train"
Xc, yt, Et, gt = DATA.loc[train, CONCEPTS].values, y_all[train], EMB[train], DATA.loc[train, "near_copy_group"].values
inv, n0, n1 = rs_table(Xc, yt)
boundary = ((n0 > 0) & (n1 > 0))[inv]
rng = np.random.default_rng(0)
same, cross = [], []
for k in np.where((n0 > 0) & (n1 > 0))[0]:
    idx = np.where(inv == k)[0]
    if len(idx) > 60:
        idx = rng.choice(idx, 60, replace=False)
    sim = Et[idx] @ Et[idx].T
    a, b = np.triu_indices(len(idx), 1)
    diff = yt[idx][a] != yt[idx][b]
    cross += sim[a, b][diff].tolist()
    same += sim[a, b][~diff].tolist()
ra, rb = rng.integers(0, train.sum(), 20000), rng.integers(0, train.sum(), 20000)
apart = inv[ra] != inv[rb]
rand = np.sum(Et[ra[apart]] * Et[rb[apart]], axis=1)
SIMS = pd.DataFrame({"pairs": ["same profile, different label", "same profile, same label", "different profiles (random)"],
                     "count": [len(cross), len(same), len(rand)],
                     "mean cosine": [np.mean(cross), np.mean(same), np.mean(rand)],
                     "median cosine": [np.median(cross), np.median(same), np.median(rand)]})
SIMS.to_csv(result("complexity_similarities.csv"), index=False)


def grouped_cv_auroc(X, y, groups, seed=0):
    s = np.zeros(len(y))
    for a, b in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(X, y, groups):
        s[b] = LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000).fit(X[a], y[a]).decision_function(X[b])
    return roc_auc_score(y, s)


Xb, yb, gb = Xc[boundary], yt[boundary], gt[boundary]
COMPLEXITY = pd.DataFrame([
    {"input": "annotated concepts", "boundary training images": int(boundary.sum()), "malignant": int(yb.sum()),
     "grouped 5-fold AUROC": grouped_cv_auroc(Xb, yb, gb), "in-sample concept ceiling AUC*": rs_stats(Xb, yb)["auroc_ceiling"]},
    {"input": f"image embeddings ({EMBEDDER})", "boundary training images": int(boundary.sum()), "malignant": int(yb.sum()),
     "grouped 5-fold AUROC": grouped_cv_auroc(Et[boundary], yb, gb), "in-sample concept ceiling AUC*": np.nan}])
COMPLEXITY.to_csv(result("complexity_test.csv"), index=False)
print(SIMS.round(4).to_string(index=False))
print(COMPLEXITY.round(4).to_string(index=False))

## Resized image cache

Every image is resized once to a short side of 400 pixels, so that decoding does not slow down training.

In [ ]:
CACHE_SHORT_SIDE = 400
SMALL_DIR = "/content/skincon_small"


def resize_one(args):
    src, dst = args
    if os.path.exists(dst):
        return
    with Image.open(src) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = CACHE_SHORT_SIDE / min(w, h)
        if s < 1:
            im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
        im.save(dst, quality=95)


os.makedirs(SMALL_DIR, exist_ok=True)
DATA["small_path"] = [os.path.join(SMALL_DIR, f"{i}.jpg") for i in DATA["image_id"]]
with ThreadPoolExecutor(IO_THREADS) as ex:
    list(tqdm(ex.map(resize_one, list(zip(DATA["path"], DATA["small_path"]))), total=len(DATA), desc="resizing"))

## Models

One concept extractor per configuration and seed is trained with the concept loss only. The three bottlenecks are
label predictors fitted afterwards on its frozen outputs, so the label loss never reaches the concepts.

- hard: a two-layer network on one-hot concept states
- soft: the same network on concept probabilities
- hybrid: a hard concept part $f$ plus $n_v$ visual units on the frozen backbone features, with logit
  $\ell(x) = f(x) + g(x)$, $g(x) = \sum_i w_i z_i(x)$, $z_i(x) \in (0, 1)$, and an L1 penalty $\lambda \|w\|_1$.
  Since $|g(x)| \le \|w\|_1$, the visual units can change a decision only when $|f(x)| \le \|w\|_1$.

In [ ]:
import copy
import glob
import dataclasses
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
TOTAL_EPOCHS = 30
RUNG_EPOCHS = 10
HEAD_HIDDEN = 128
MODEL_KINDS = ("hard", "soft", "hybrid")
TRIALS_DIR = os.path.join(WORK_DIR, "trials")
MODELS_DIR = os.path.join(WORK_DIR, "models")
SELECTION_FILE = os.path.join(WORK_DIR, "selection.json")
for d in (TRIALS_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)


@dataclasses.dataclass
class TrialConfig:
    name: str
    tunable_pct: int = 75            # share of backbone blocks that are trained, from the top
    batch_size: int = 16
    lr: float = 5e-4                 # learning rate of the concept heads
    backbone_lr_mult: float = 1.0    # backbone learning rate = lr * backbone_lr_mult
    optimizer: str = "adam"
    weight_decay: float = 0.0
    head_dropout: float = 0.0
    label_smoothing: float = 0.0
    stochastic_depth: float = 0.2
    augment: str = "standard"        # standard or strong
    img_size: int = 224
    warmup_epochs: int = 0
    concept_weighting: bool = True   # class weights in the concept loss


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    os.environ["PYTHONHASHSEED"] = str(seed)


BACKBONES = {"efficientnet_b0": ("efficientnet_b0", "EfficientNet_B0_Weights", 1280),
             "efficientnet_b3": ("efficientnet_b3", "EfficientNet_B3_Weights", 1536),
             "efficientnet_b4": ("efficientnet_b4", "EfficientNet_B4_Weights", 1792)}


class ConceptModel(nn.Module):
    def __init__(self, cfg, concepts, pretrained=True):
        super().__init__()
        fn, weights, dim = BACKBONES[BACKBONE_NAME]
        m = getattr(models, fn)(weights=getattr(models, weights).DEFAULT if pretrained else None,
                                stochastic_depth_prob=cfg.stochastic_depth)
        self.features = m.features
        frozen = int(round(len(self.features) * (100 - cfg.tunable_pct) / 100))
        for i, layer in enumerate(self.features):
            for p in layer.parameters():
                p.requires_grad = i >= frozen
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(cfg.head_dropout)
        self.concepts = list(concepts)
        self.heads = nn.ModuleDict({c: nn.Linear(dim, 2) for c in self.concepts})

    def embed(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x, return_features=False):
        e = self.embed(x)
        logits = [self.heads[c](self.drop(e)) for c in self.concepts]
        return (logits, e) if return_features else logits


class SkinConDataset(Dataset):
    def __init__(self, df, concepts, transform):
        self.paths = df["small_path"].tolist()
        self.c = torch.tensor(df[concepts].values, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        return self.transform(img), self.c[i]


NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])


def train_transforms(cfg):
    t = [transforms.RandomResizedCrop(cfg.img_size, scale=(0.7 if cfg.augment == "strong" else 0.8, 1.0)),
         transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(90),
         transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15)], p=0.5),
         transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
         transforms.RandAugment(num_ops=2, magnitude=12 if cfg.augment == "strong" else 9),
         transforms.ToTensor(), NORM]
    if cfg.augment == "strong":
        t.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    return transforms.Compose(t)


def eval_transforms(cfg):
    return transforms.Compose([transforms.Resize((cfg.img_size, cfg.img_size)), transforms.ToTensor(), NORM])


def concept_weights(df, concepts):
    out = []
    for c in concepts:
        pos = min(max(float(df[c].mean()), 1e-3), 1 - 1e-3)
        out.append(torch.tensor([0.5 / (1 - pos), 0.5 / pos], dtype=torch.float32, device=DEVICE))
    return out


def make_optimizer(cfg, model):
    backbone = [p for p in model.features.parameters() if p.requires_grad]
    groups = [{"params": backbone, "lr": cfg.lr * cfg.backbone_lr_mult}, {"params": list(model.heads.parameters()), "lr": cfg.lr}]
    if cfg.optimizer == "adamw":
        return optim.AdamW(groups, weight_decay=cfg.weight_decay)
    return optim.Adam(groups, weight_decay=cfg.weight_decay)


def lr_factor(cfg, epoch):
    """Linear warmup, then cosine decay to 2% of the base rate."""
    if epoch < cfg.warmup_epochs:
        return (epoch + 1) / (cfg.warmup_epochs + 1)
    t = (epoch - cfg.warmup_epochs) / max(1, TOTAL_EPOCHS - cfg.warmup_epochs)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))


@torch.no_grad()
def concept_probs(model, df, cfg, batch=64):
    """Concept probabilities and pooled backbone features of every image of df."""
    model.eval()
    dl = DataLoader(SkinConDataset(df, CONCEPTS, eval_transforms(cfg)), batch_size=batch, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    probs, feats = [], []
    for x, _ in dl:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            lg, e = model(x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last), return_features=True)
        probs.append(torch.stack([torch.softmax(l.float(), 1)[:, 1] for l in lg], 1).cpu().numpy())
        feats.append(e.float().cpu().numpy())
    return np.concatenate(probs).astype(np.float64), np.concatenate(feats).astype(np.float32)


def concept_metrics(prob, df, thresholds=None):
    t = df[CONCEPTS].values
    thr = np.full(len(CONCEPTS), 0.5) if thresholds is None else np.asarray(thresholds)
    hard = (prob >= thr[None, :]).astype(int)
    ok = [0 < t[:, j].sum() < len(t) for j in range(len(CONCEPTS))]
    auc = [roc_auc_score(t[:, j], prob[:, j]) if ok[j] else np.nan for j in range(len(CONCEPTS))]
    f1 = [f1_score(t[:, j], hard[:, j], average="macro", zero_division=0) for j in range(len(CONCEPTS))]
    ba = [balanced_accuracy_score(t[:, j], hard[:, j]) if ok[j] else np.nan for j in range(len(CONCEPTS))]
    return {"concept_macro_auroc": float(np.nanmean(auc)), "concept_macro_f1": float(np.mean(f1)),
            "concept_balanced_accuracy": float(np.nanmean(ba)), "exact_profile_rate": float((hard == t).all(axis=1).mean())}


def validation_thresholds(prob, df):
    """Per-concept threshold that maximizes the balanced accuracy on the given images."""
    t = df[CONCEPTS].values
    thr = np.full(len(CONCEPTS), 0.5)
    grid = np.linspace(0.05, 0.95, 91)
    for j in range(len(CONCEPTS)):
        if 0 < t[:, j].sum() < len(t):
            thr[j] = grid[int(np.argmax([balanced_accuracy_score(t[:, j], (prob[:, j] >= g).astype(int)) for g in grid]))]
    return thr


def one_hot(states):
    n_img, n = states.shape
    X = np.zeros((n_img, 2 * n))
    X[np.arange(n_img)[:, None], 2 * np.arange(n)[None, :] + states] = 1.0
    return X


def make_head(kind, n, dropout=0.3):
    d = n if kind == "soft" else 2 * n
    return nn.Sequential(nn.Linear(d, HEAD_HIDDEN), nn.LayerNorm(HEAD_HIDDEN), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(HEAD_HIDDEN, 2))


class HybridHead(nn.Module):
    def __init__(self, n, feat_dim, n_visual, dropout=0.3):
        super().__init__()
        self.concept = make_head("hard", n, dropout)
        self.norm = nn.LayerNorm(feat_dim)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(feat_dim, n_visual)
        self.w = nn.Linear(n_visual, 1, bias=False)

    def parts(self, xc, xf):
        o = self.concept(xc)
        z = torch.sigmoid(self.proj(self.drop(self.norm(xf))))
        return o, self.w(z).squeeze(1)

    def forward(self, xc, xf):
        o, g = self.parts(xc, xf)
        return torch.stack([o[:, 0], o[:, 1] + g], 1)

    def l1(self):
        return self.w.weight.abs().sum()


def head_inputs(kind, states=None, probs=None, feats=None):
    if kind == "hard":
        return (torch.tensor(one_hot(states), dtype=torch.float32),)
    if kind == "soft":
        return (torch.tensor(probs, dtype=torch.float32),)
    return torch.tensor(one_hot(states), dtype=torch.float32), torch.tensor(feats, dtype=torch.float32)


def build_head(kind, n, feat_dim=None, n_visual=0, dropout=0.3):
    return HybridHead(n, feat_dim, n_visual, dropout) if kind == "hybrid" else make_head(kind, n, dropout)


def fit_head(kind, tr_in, ytr, va_in, yva, seed, n, feat_dim=None, n_visual=0, l1=0.0, weight_decay=1e-4,
             lr=1e-3, batch=32, dropout=0.3, epochs=300, patience=40):
    """Class-weighted cross-entropy plus l1 * ||w||_1 for the hybrid, early stopping on the validation loss."""
    set_seed(seed)
    yt, yv = torch.tensor(ytr, dtype=torch.long), torch.tensor(yva, dtype=torch.long)
    cw = torch.tensor(len(yt) / (2.0 * np.bincount(ytr, minlength=2).clip(1)), dtype=torch.float32)
    head = build_head(kind, n, feat_dim, n_visual, dropout)
    opt = optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    g = torch.Generator()
    g.manual_seed(seed)
    best, state, best_epoch, wait = np.inf, None, 0, 0
    for epoch in range(1, epochs + 1):
        head.train()
        perm = torch.randperm(len(yt), generator=g)
        for i in range(0, len(yt), batch):
            b = perm[i:i + batch]
            opt.zero_grad()
            loss = F.cross_entropy(head(*[t[b] for t in tr_in]), yt[b], weight=cw)
            if kind == "hybrid" and l1 > 0:
                loss = loss + l1 * head.l1()
            loss.backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val = F.cross_entropy(head(*va_in), yv, weight=cw).item()
        if val < best - 1e-6:
            best, state, best_epoch, wait = val, copy.deepcopy(head.state_dict()), epoch, 0
        else:
            wait += 1
            if wait >= patience:
                break
    head.load_state_dict(state)
    head.eval()
    return head, best_epoch


def head_logit(head, inputs):
    with torch.no_grad():
        o = head(*inputs)
    return (o[:, 1] - o[:, 0]).double().numpy()


def hybrid_parts(head, inputs):
    """Concept part f, visual part g, ||w||_1, the visual share and the share of decisions changed by g."""
    with torch.no_grad():
        o, g = head.parts(*inputs)
    f = (o[:, 1] - o[:, 0]).double().numpy()
    g = g.double().numpy()
    bound = float(head.w.weight.abs().sum())
    return {"f": f, "g": g, "w_l1": bound, "visual_share": float(np.abs(g).mean() / (np.abs(f) + np.abs(g)).mean()),
            "flip_possible": float((np.abs(f) <= bound).mean()), "flipped": float((np.sign(f) != np.sign(f + g)).mean())}


def label_metrics(y, s):
    pred = (s > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"), "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1), "auroc": roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan}


def trial_dir(cfg, seed):
    d = os.path.join(TRIALS_DIR, f"{cfg.name}_seed{seed}")
    os.makedirs(d, exist_ok=True)
    return d


def run_trial(cfg, seed, until_epoch, splits):
    """Train the concept extractor up to until_epoch of the schedule, resuming from the last checkpoint.
    best.pth keeps the weights with the best validation concept AUROC."""
    d = trial_dir(cfg, seed)
    last, best_path, hist_path = (os.path.join(d, f) for f in ("last.pth", "best.pth", "history.csv"))
    with open(os.path.join(d, "config.json"), "w") as fh:
        json.dump({"seed": seed, **dataclasses.asdict(cfg)}, fh, indent=1)
    if not os.path.exists(last) and os.path.exists(best_path) and os.path.exists(hist_path):
        if len(pd.read_csv(hist_path)) >= until_epoch:
            return
    train, valid = splits["train"], splits["valid"]
    set_seed(seed)
    model = ConceptModel(cfg, CONCEPTS).to(DEVICE).to(memory_format=torch.channels_last)
    opt = make_optimizer(cfg, model)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    start, best_auc, best_epoch, history = 0, -1.0, -1, []
    if os.path.exists(last):
        ck = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best_auc, best_epoch, history = ck["epoch"], ck["best_auc"], ck["best_ep"], ck["history"]
        torch.set_rng_state(ck["rng_cpu"].cpu())
        if torch.cuda.is_available() and ck.get("rng_cuda") is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in ck["rng_cuda"]])
    if start >= until_epoch:
        return
    g = torch.Generator()
    g.manual_seed(seed * 1000 + start)
    dl = DataLoader(SkinConDataset(train, CONCEPTS, train_transforms(cfg)), batch_size=cfg.batch_size, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, generator=g, persistent_workers=False)
    weights = concept_weights(train, CONCEPTS) if cfg.concept_weighting else [None] * len(CONCEPTS)
    base_lrs = [cfg.lr * cfg.backbone_lr_mult, cfg.lr]
    for epoch in range(start, until_epoch):
        t0 = time.time()
        for group, base in zip(opt.param_groups, base_lrs):
            group["lr"] = base * lr_factor(cfg, epoch)
        model.train()
        total, batches = 0.0, 0
        for x, c in tqdm(dl, desc=f"{cfg.name} seed {seed} epoch {epoch + 1}", leave=False):
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            c = c.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                lg = model(x)
                loss = torch.stack([F.cross_entropy(lg[j].float(), c[:, j], weight=weights[j],
                                                    label_smoothing=cfg.label_smoothing)
                                    for j in range(len(CONCEPTS))]).mean()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += loss.item()
            batches += 1
        pva, _ = concept_probs(model, valid, cfg)
        m = concept_metrics(pva, valid)
        history.append({"epoch": epoch + 1, "train_loss": total / max(batches, 1), "seconds": time.time() - t0, **m})
        if m["concept_macro_auroc"] > best_auc:
            best_auc, best_epoch = m["concept_macro_auroc"], epoch + 1
            torch.save(model.state_dict(), best_path)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch + 1, "best_auc": best_auc, "best_ep": best_epoch, "history": history,
                    "rng_cpu": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, last)
        pd.DataFrame(history).to_csv(hist_path, index=False)
        print(f"{cfg.name} seed {seed} epoch {epoch + 1}/{TOTAL_EPOCHS}: loss {history[-1]['train_loss']:.4f}, "
              f"validation concept AUROC {m['concept_macro_auroc']:.4f}")
    del model, opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_best(cfg, seed):
    model = ConceptModel(cfg, CONCEPTS, pretrained=False).to(DEVICE).to(memory_format=torch.channels_last)
    model.load_state_dict(torch.load(os.path.join(trial_dir(cfg, seed), "best.pth"), map_location=DEVICE, weights_only=False))
    return model


def extractor_outputs(cfg, seed, splits):
    """Concept probabilities and backbone features of the best weights, cached next to the checkpoint."""
    d = trial_dir(cfg, seed)
    cache = os.path.join(d, "outputs.npz")
    if os.path.exists(cache) and os.path.getmtime(cache) >= os.path.getmtime(os.path.join(d, "best.pth")):
        z = np.load(cache)
        return {s: z[f"prob_{s}"] for s in splits}, {s: z[f"feat_{s}"] for s in splits}
    model = load_best(cfg, seed)
    probs, feats = {}, {}
    for s in splits:
        probs[s], feats[s] = concept_probs(model, splits[s], cfg)
    np.savez_compressed(cache, **{f"prob_{s}": probs[s] for s in splits}, **{f"feat_{s}": feats[s].astype(np.float16) for s in splits})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return probs, {s: feats[s].astype(np.float32) for s in splits}


def evaluate_trial(cfg, seed, splits, stage, extra=None):
    """Validation scores of the extractor and of a hard predictor with default settings."""
    probs, _ = extractor_outputs(cfg, seed, splits)
    states = {s: (probs[s] >= 0.5).astype(int) for s in probs}
    head, _ = fit_head("hard", head_inputs("hard", states["train"]), splits["train"]["malignant"].values,
                       head_inputs("hard", states["valid"]), splits["valid"]["malignant"].values, seed, len(CONCEPTS))
    lab = label_metrics(splits["valid"]["malignant"].values, head_logit(head, head_inputs("hard", states["valid"])))
    row = {"stage": stage, "config": cfg.name, "seed": seed, **(extra or {}),
           **{f"valid_{k}": v for k, v in concept_metrics(probs["valid"], splits["valid"]).items()},
           **{f"valid_hard_{k}": v for k, v in lab.items()}}
    return row


def load_selection():
    if os.path.exists(SELECTION_FILE):
        with open(SELECTION_FILE) as fh:
            return json.load(fh)
    return {}


def save_selection(**kw):
    d = load_selection()
    d.update(kw)
    with open(SELECTION_FILE, "w") as fh:
        json.dump(d, fh, indent=1, default=str)


SPLITS = {s: DATA[DATA["split"] == s].reset_index(drop=True) for s in ("train", "valid", "test")}
print(f"device {DEVICE}, {len(SPLITS['train'])} training images")

## Stage A, concept extractor search on seed 42

The base configuration trains the whole backbone with AdamW (learning rate 3e-4, backbone multiplier 0.3, weight decay
0.05), batches of 8 images of 300 pixels, dropout 0.5, stochastic depth 0.4 and strong augmentation. Each variant changes
one factor. Four configurations train for 10 of 30 epochs, ranked by the validation AUROC of a hard predictor with
default settings, and the best two continue to 30 epochs.

In [ ]:
BASE = TrialConfig("base", tunable_pct=100, batch_size=8, lr=3e-4, backbone_lr_mult=0.3, optimizer="adamw",
                   weight_decay=0.05, head_dropout=0.5, label_smoothing=0.0, stochastic_depth=0.4, augment="strong",
                   img_size=300, warmup_epochs=1, concept_weighting=False)
CONFIGS = [BASE,
           dataclasses.replace(BASE, name="base_cw", concept_weighting=True),
           dataclasses.replace(BASE, name="base_lr1e-3", lr=1e-3),
           dataclasses.replace(BASE, name="base_bb1.0", backbone_lr_mult=1.0)]
CFG_BY_NAME = {c.name: c for c in CONFIGS}
SEARCH_SEED = SEEDS[0]
RANK = ["valid_hard_auroc", "valid_concept_macro_auroc"]
SHOW = ["config", "valid_hard_auroc", "valid_hard_balanced_accuracy", "valid_concept_macro_auroc",
        "valid_concept_macro_f1", "valid_exact_profile_rate"]

rung1 = []
for cfg in CONFIGS:
    run_trial(cfg, SEARCH_SEED, RUNG_EPOCHS, SPLITS)
    rung1.append(evaluate_trial(cfg, SEARCH_SEED, SPLITS, "A_rung1", {"epochs": RUNG_EPOCHS}))
RUNG1 = pd.DataFrame(rung1).sort_values(RANK, ascending=False)
RUNG1.to_csv(result("stageA_rung1.csv"), index=False)
print(RUNG1[SHOW].round(4).to_string(index=False))

promoted = load_selection().get("promoted") or RUNG1["config"].head(2).tolist()
save_selection(promoted=promoted)
rung2 = []
for name in promoted:
    run_trial(CFG_BY_NAME[name], SEARCH_SEED, TOTAL_EPOCHS, SPLITS)
    rung2.append(evaluate_trial(CFG_BY_NAME[name], SEARCH_SEED, SPLITS, "A_rung2", {"epochs": TOTAL_EPOCHS}))
RUNG2 = pd.DataFrame(rung2).sort_values(RANK, ascending=False)
RUNG2.to_csv(result("stageA_rung2.csv"), index=False)
pd.DataFrame(rung1 + rung2).to_csv(result("trials.csv"), index=False)
BEST_NAME = load_selection().get("best_config") or RUNG2.iloc[0]["config"]
save_selection(best_config=BEST_NAME)
FINAL = CFG_BY_NAME[BEST_NAME]
print(RUNG2[SHOW].round(4).to_string(index=False))
print(f"selected concept extractor: {BEST_NAME}")
for cfg in CONFIGS:
    f = os.path.join(trial_dir(cfg, SEARCH_SEED), "last.pth")
    if cfg.name not in promoted and os.path.exists(f):
        os.remove(f)

## Stage B, label predictors on validation images

With the seed-42 extractor, the hard predictor chooses the state rule (threshold 0.5 or per-concept validation
thresholds), the weight decay and the learning rate. The soft predictor chooses the last two. The hybrid predictor
keeps the best hard settings and chooses $n_v$ and $\lambda$ among the settings whose visual units carry at most 30% of
the validation logit.

In [ ]:
probs42, feats42 = extractor_outputs(FINAL, SEARCH_SEED, SPLITS)
Y = {s: SPLITS[s]["malignant"].values for s in SPLITS}
NC, FEAT_DIM = len(CONCEPTS), feats42["train"].shape[1]
STATE_RULES = {"argmax": np.full(NC, 0.5), "validation_threshold": validation_thresholds(probs42["valid"], SPLITS["valid"])}


def inputs_for(kind, probs, feats, thr, split):
    return head_inputs(kind, (probs[split] >= thr).astype(int), probs[split], feats[split])


def score_head(kind, opts, probs, feats, seed):
    thr = STATE_RULES[opts.get("states", "argmax")]
    tr_in, va_in = inputs_for(kind, probs, feats, thr, "train"), inputs_for(kind, probs, feats, thr, "valid")
    head, epoch = fit_head(kind, tr_in, Y["train"], va_in, Y["valid"], seed, NC, FEAT_DIM, opts.get("n_visual", 0),
                           opts.get("l1", 0.0), opts["weight_decay"], opts["lr"], 32)
    row = {"model": kind, **opts, "best_epoch": epoch,
           **{f"valid_{k}": v for k, v in label_metrics(Y["valid"], head_logit(head, va_in)).items()}}
    if kind == "hybrid":
        parts = hybrid_parts(head, va_in)
        row.update({k: parts[k] for k in ("w_l1", "visual_share", "flip_possible", "flipped")})
    return row


SETTINGS = [("hard", {"states": st, "weight_decay": wd, "lr": lr})
            for st in STATE_RULES for wd in (1e-4, 1e-2) for lr in (1e-3, 3e-3)]
SETTINGS += [("soft", {"weight_decay": wd, "lr": lr}) for wd in (1e-4, 1e-2) for lr in (1e-3, 3e-3)]
grid = [score_head(kind, opts, probs42, feats42, SEARCH_SEED) for kind, opts in tqdm(SETTINGS, desc="hard and soft")]
best_hard = pd.DataFrame(grid).query("model == 'hard'").sort_values("valid_auroc", ascending=False).iloc[0]
for nv, l1 in tqdm([(nv, l1) for nv in (8, 32) for l1 in (1e-3, 1e-2, 1e-1)], desc="hybrid"):
    opts = {"states": best_hard["states"], "weight_decay": float(best_hard["weight_decay"]),
            "lr": float(best_hard["lr"]), "n_visual": nv, "l1": l1}
    grid.append(score_head("hybrid", opts, probs42, feats42, SEARCH_SEED))
HEAD_GRID = pd.DataFrame(grid)
HEAD_GRID.to_csv(result("stageB_heads.csv"), index=False)
print(HEAD_GRID.sort_values(["model", "valid_auroc"], ascending=[True, False]).round(4).to_string(index=False))


def pick(kind):
    g = HEAD_GRID[HEAD_GRID.model == kind]
    if kind == "hybrid":
        ok = g[g.visual_share <= MAX_VISUAL_SHARE]
        g = ok if len(ok) else g.sort_values("visual_share").head(1)
    r = g.sort_values("valid_auroc", ascending=False).iloc[0]
    keys = {"hard": ("states", "weight_decay", "lr"), "soft": ("weight_decay", "lr"),
            "hybrid": ("states", "weight_decay", "lr", "n_visual", "l1")}[kind]
    out = {k: (r[k].item() if hasattr(r[k], "item") else r[k]) for k in keys}
    if "n_visual" in out:
        out["n_visual"] = int(out["n_visual"])
    return out


if "head_options" not in load_selection():
    save_selection(head_options={k: pick(k) for k in MODEL_KINDS})
HOPT = load_selection()["head_options"]
pd.DataFrame.from_dict(HOPT, orient="index").rename_axis("model").to_csv(result("selected_heads.csv"))
print(pd.read_csv(result("selected_heads.csv")).to_string(index=False))

## Stage C, final models

The selected extractor is trained with the three seeds (the seed-42 run is reused) and each seed gets the three label
predictors. The test split is used once, for the final report. Each seed is saved as one file
`model_efficientnet_b3_seed<seed>.pth` that holds the concept extractor and the three predictors.

In [ ]:
metric_rows, prediction_frames = [], []
for seed in SEEDS:
    run_trial(FINAL, seed, TOTAL_EPOCHS, SPLITS)
    probs, feats = extractor_outputs(FINAL, seed, SPLITS)
    thr_seed = validation_thresholds(probs["valid"], SPLITS["valid"])
    for s in SPLITS:
        metric_rows.append({"seed": seed, "model": "concepts", "split": s, "source": "all",
                            **concept_metrics(probs[s], SPLITS[s], thr_seed)})
    heads = {}
    for kind in MODEL_KINDS:
        o = HOPT[kind]
        thr = thr_seed if o.get("states") == "validation_threshold" else np.full(NC, 0.5)
        ins = {s: inputs_for(kind, probs, feats, thr, s) for s in SPLITS}
        head, epoch = fit_head(kind, ins["train"], Y["train"], ins["valid"], Y["valid"], seed, NC, FEAT_DIM,
                               o.get("n_visual", 0), o.get("l1", 0.0), o["weight_decay"], o["lr"], 32)
        heads[kind] = {"kind": kind, "options": o, "thresholds": thr, "best_epoch": epoch,
                       "state_dict": head.state_dict(), "feat_dim": FEAT_DIM}
        for s in SPLITS:
            score = head_logit(head, ins[s])
            df = SPLITS[s]
            parts = hybrid_parts(head, ins[s]) if kind == "hybrid" else {}
            for src in ("all",) + tuple(sorted(df["source"].unique())):
                m = np.ones(len(df), bool) if src == "all" else (df["source"] == src).values
                metric_rows.append({"seed": seed, "model": kind, "split": s, "source": src, "best_epoch": epoch,
                                    **label_metrics(Y[s][m], score[m]),
                                    **({k: parts[k] for k in ("visual_share", "flip_possible", "flipped", "w_l1")} if parts else {})})
            prediction_frames.append(pd.DataFrame({
                "seed": seed, "model": kind, "split": s, "image_id": df["image_id"], "source": df["source"],
                "diagnosis": df["diagnosis"], "partition": df["partition"], "y_true": Y[s], "logit": score,
                "y_pred": (score > 0).astype(int),
                **({"concept_part": parts["f"], "visual_part": parts["g"]} if parts else {})}))
    torch.save({"backbone": BACKBONE_NAME, "seed": seed, "config": dataclasses.asdict(FINAL), "concepts": CONCEPTS,
                "concept_names": CONCEPT_NAMES, "heads": heads,
                "concept_model": torch.load(os.path.join(trial_dir(FINAL, seed), "best.pth"), map_location="cpu", weights_only=False)},
               os.path.join(MODELS_DIR, f"model_{BACKBONE_NAME}_seed{seed}.pth"))
METRICS = pd.DataFrame(metric_rows)
METRICS.to_csv(result("training_metrics.csv"), index=False)
PREDICTIONS = pd.concat(prediction_frames, ignore_index=True)
PREDICTIONS.to_csv(result(f"predictions_{BACKBONE_NAME}.csv"), index=False)

## Non-malignant diagnoses that the models flag

False-positive rate of every non-malignant diagnosis with at least five test images, averaged over the seeds, with a
Wilson interval over the pooled seeds. The forced rate is the share of the diagnosis that the balanced concept-only rule
on annotated concepts labels malignant.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(c - h, 0.0), min(c + h, 1.0)


negatives = PREDICTIONS[(PREDICTIONS.split == "test") & (PREDICTIONS.y_true == 0)]
conflict = DATA.set_index("image_id")["diagnosis_conflict"].astype(bool)
negatives = negatives[~negatives["image_id"].map(conflict).fillna(False).values]
negatives = negatives.assign(forced=negatives["image_id"].map(DATA.set_index("image_id")["forced_error"]).astype(float))
per_seed = negatives.groupby(["model", "diagnosis", "seed"]).agg(images=("y_pred", "size"), fp=("y_pred", "sum"))
per_seed["fp_rate"] = per_seed["fp"] / per_seed["images"]
agg = per_seed.groupby(["model", "diagnosis"]).agg(test_images=("images", "first"), fp_total=("fp", "sum"),
                                                   seed_draws=("images", "sum"), fp_rate=("fp_rate", "mean"),
                                                   fp_rate_std=("fp_rate", "std")).reset_index()
ci = [wilson(k, n) for k, n in zip(agg.fp_total, agg.seed_draws)]
agg["ci_low"], agg["ci_high"] = [c[0] for c in ci], [c[1] for c in ci]
agg["partition"] = agg["diagnosis"].map(DATA.drop_duplicates("diagnosis").set_index("diagnosis")["partition"])
agg["forced_fp_rate"] = agg["diagnosis"].map(DATA[DATA.malignant == 0].groupby("diagnosis")["forced_error"].mean())
FLAGGED = agg[agg.test_images >= 5].sort_values(["model", "fp_rate"], ascending=[True, False])
FLAGGED.to_csv(result("misclassified_non_malignant.csv"), index=False)
for kind in MODEL_KINDS:
    t = FLAGGED[FLAGGED.model == kind]
    rho = spearmanr(t["fp_rate"], t["forced_fp_rate"], nan_policy="omit")
    print(f"{kind}: Spearman correlation between the false-positive rate and the forced rate "
          f"{rho.correlation:.3f} (p = {rho.pvalue:.3g}, {len(t)} diagnoses)")
    print(t[["diagnosis", "partition", "test_images", "fp_rate", "ci_low", "ci_high", "forced_fp_rate"]].head(10).round(3).to_string(index=False))

## Test summary and training histories

In [ ]:
cols = ["auroc", "balanced_accuracy", "macro_f1", "sensitivity", "specificity"]
t = METRICS[(METRICS.model != "concepts") & (METRICS.split == "test")]
SUMMARY = t.groupby(["model", "source"])[cols].agg(["mean", "std"])
SUMMARY.columns = [f"{a}_{b}" for a, b in SUMMARY.columns]
SUMMARY = SUMMARY.reset_index()
SUMMARY.to_csv(result("summary_test.csv"), index=False)
print(SUMMARY.round(4).to_string(index=False))
concepts_test = METRICS[(METRICS.model == "concepts") & (METRICS.split == "test")]
print(concepts_test[["concept_macro_auroc", "concept_macro_f1", "concept_balanced_accuracy", "exact_profile_rate"]].agg(["mean", "std"]).round(4))


def export_trials():
    histories, configs = [], []
    for d in sorted(glob.glob(os.path.join(TRIALS_DIR, "*"))):
        with open(os.path.join(d, "config.json")) as fh:
            cfg = json.load(fh)
        h = pd.read_csv(os.path.join(d, "history.csv"))
        h.insert(0, "seed", cfg["seed"])
        h.insert(0, "config", cfg["name"])
        histories.append(h)
        configs.append({"config": cfg["name"], **{k: v for k, v in cfg.items() if k != "name"}})
    pd.concat(histories, ignore_index=True).to_csv(result("training_histories.csv"), index=False)
    pd.DataFrame(configs).to_csv(result("trial_configs.csv"), index=False)


export_trials()

## Using a saved model

`predict_images` returns the concept probabilities, the concept states and the probability of malignancy for a list of
image files. Run the model cell first, since it defines the classes.

In [ ]:
@torch.no_grad()
def predict_images(bundle_path, image_paths, model_kind="hard"):
    b = torch.load(bundle_path, map_location="cpu", weights_only=False)
    cfg = TrialConfig(**b["config"])
    model = ConceptModel(cfg, b["concepts"], pretrained=False)
    model.load_state_dict(b["concept_model"])
    model.to(DEVICE).eval()
    tf = eval_transforms(cfg)
    x = torch.stack([tf(Image.open(p).convert("RGB")) for p in image_paths]).to(DEVICE)
    lg, e = model(x, return_features=True)
    prob = torch.stack([torch.softmax(l.float(), 1)[:, 1] for l in lg], 1).cpu().numpy()
    h = b["heads"][model_kind]
    states = (prob >= np.asarray(h["thresholds"])[None, :]).astype(int)
    net = build_head(model_kind, len(b["concepts"]), h["feat_dim"], h["options"].get("n_visual", 0))
    net.load_state_dict(h["state_dict"])
    net.eval()
    logit = head_logit(net, head_inputs(model_kind, states, prob, e.float().cpu().numpy()))
    return {"concepts": b["concepts"], "concept_probs": prob, "concept_states": states,
            "malignant_probability": 1 / (1 + np.exp(-logit))}


# out = predict_images(f"{MODELS_DIR}/model_{BACKBONE_NAME}_seed42.pth", ["/content/example.jpg"], "hybrid")